<a href="https://colab.research.google.com/github/skwent77/CodingPractice/blob/main/Cohere_Reranker%26RAG_with_chroma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU cohere langchain langchain-cohere langchain_community faiss-cpu docling \
langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.7/814.7 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [3]:
import langchain
import chromadb
import sentence_transformers

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("정상 import 완료")

정상 import 완료


### 🗝️ Cohere API Key

https://dashboard.cohere.com/api-keys

In [4]:
import os

os.environ["COHERE_API_KEY"] = "rijglQ78JwINO6USIAlBfVOOWu1buDXfuhW8jTBw" # 발급받은 API Key 를 복사 + 붙여넣기

RAG 시대, Vector DB 비교
cypher는 그래프를 위한 sql이고 Neo4j는 그래프 데이터 시각화 위한 툴

Langchain의 component 8개

## Cohere Embedding model - Retriever

#### 0. Retriever에 사용할 임베딩 모델

- Cohere Embed Models : https://docs.cohere.com/docs/cohere-embed

In [5]:
from langchain_cohere import CohereEmbeddings

cohere_embeddings = CohereEmbeddings(model="embed-english-light-v3.0") # 다국어 : embed-multilingual-light-v3.0, ...
text = "This is a test document."

embedding_result = cohere_embeddings.embed_query(text)
print(embedding_result)

[-0.09301758, 0.08703613, -0.033294678, 0.021133423, 0.07678223, 0.033966064, -0.05911255, -0.030975342, -0.030090332, -0.05633545, 0.050933838, -0.006450653, 0.038513184, -0.0010023117, -0.041534424, -0.009414673, 0.02684021, -0.0234375, 0.055267334, -0.032714844, 0.055603027, 0.028167725, -0.06945801, 0.03955078, -0.0670166, 0.066589355, -0.012435913, -0.0070724487, 0.03793335, 0.058044434, 0.047454834, 0.02720642, 0.01033783, 0.08270264, 0.073791504, 0.016983032, -0.08685303, 0.017562866, -0.0025615692, -0.0038661957, 0.040649414, -0.030578613, 0.009269714, 0.011451721, 0.0077209473, 0.09063721, 0.052703857, -0.061309814, 0.03878784, -0.018295288, -0.04550171, -0.08319092, 0.0071640015, -0.0602417, 0.0024280548, 0.050689697, 0.052001953, -0.046905518, -0.051483154, -0.04458618, -0.05770874, 0.07940674, -0.019332886, 0.06427002, 0.025131226, 0.011116028, -0.037384033, -0.01890564, -0.06762695, -0.052703857, -0.018997192, -0.020874023, 0.044281006, 0.07495117, 0.08282471, 0.0025920868

In [6]:
print("Embedding Dim :",len(embedding_result))

Embedding Dim : 384


embedding모델 기반으로 BaseRetriver 구현

한글로 테스트

In [ ]:
%cd /content/drive/sample.txt

#### 1. 샘플 텍스트 Load 하기

- Sample txt(Kor) : https://n.news.naver.com/mnews/article/003/0013004563

In [11]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = TextLoader("sample.txt").load()

In [ ]:
documents

#### 2. 텍스트 Split 하기

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=90, chunk_overlap=20)
#중첩되는 무부분은 문자 50개까지는 겹쳐도 되게
texts = text_splitter.split_documents(documents)

In [13]:
texts

[Document(metadata={'source': 'sample.txt'}, page_content='경기도 전역 한파특보…재난안전 1단계 대응\n이병희 기자\n\n인쇄하기\n8일 오후 9시 31개 시군에 한파특보 발효 예정'),
 Document(metadata={'source': 'sample.txt'}, page_content='[수원=뉴시스] 이병희 기자 = 경기도가 도내 31개 시군 전역에 예보된 한파특보에 따라 8일 오후 1시부터 재난안전대책본부 1단계를 가동, 선제 대응에'),
 Document(metadata={'source': 'sample.txt'}, page_content='1단계를 가동, 선제 대응에 나섰다.'),
 Document(metadata={'source': 'sample.txt'}, page_content='기상청은 이날 오후 9시를 기해 경기도 전역에 한파특보를 발효할 예정이다. 한파주의보가 내려졌던 연천·포천·가평·파주 등 4개 시·군은 한파경보로 변경되고,'),
 Document(metadata={'source': 'sample.txt'}, page_content='4개 시·군은 한파경보로 변경되고, 동두천·양주·의정부·남양주·여주·양평 등 6개 시·군은 한파경보가 발효된다.'),
 Document(metadata={'source': 'sample.txt'}, page_content='또 광명, 과천, 안산, 시흥, 부천, 김포, 고양, 수원, 성남, 안양, 구리, 오산, 평택, 군포, 의왕, 하남, 용인, 이천, 안성, 화성, 광주 등'),
 Document(metadata={'source': 'sample.txt'}, page_content='이천, 안성, 화성, 광주 등 경기남부지역을 비롯한 21개 지역에는 한파주의보를 발표됐다.'),
 Document(metadata={'source': 'sample.txt'}, page_content='북서쪽에서 남하하는 찬 공기의 영향으로 9~10일 아침 기온은 이날보다 5~

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings  # latest
from langchain_chroma import Chroma

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2" # 유사한 키워드는 유사한 숫자로 만들어 젖장

#### 3. FAISS (Vector Store) 적재하기 -> Chroma 적재하기


1.   document형식인데 이걸 벡터 스토어로 저장 (langchain의 docs 타입은 뭐로 나와)
2.   벡터 스토어 chunks 추가 후 질문으로 similary search



In [ ]:
# from langchain_community.vectorstores import FAISS

# Face book ai similary search 문서의 임베딩 벡터들을 저장하는 라이블러ㅣ
from langchain_cohere import CohereEmbeddings

retriever = FAISS.from_documents(
    texts, CohereEmbeddings(model="embed-multilingual-v3.0")
).as_retriever(search_kwargs={"k": 10})

In [15]:
# instead

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="news"
)
# retriever = FAISS.from_documents(
#     texts, CohereEmbeddings(model="embed-multilingual-v3.0")
# ).as_retriever(search_kwargs={"k": 10})
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

Document 객체는 page_content , metadata two attributes

In [20]:
print(type(texts))
print(type(texts[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


#### 4. Retriever 생성 및 문서 검색하기
관련성 평가 기준: 임베딩

In [17]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [21]:
query = "한파에 대비해야할 것에는 어떤게 있나요?"
docs = retriever.invoke(query)
pretty_print_docs(docs)

Document 1:

도는 재난안전대책본부 1단계를 가동해 복지·상하수 분야 등 6개 반 13개 부서 13명이 분야별 대응하며 한파에 대비할 방침이다.
----------------------------------------------------------------------------------------------------
Document 2:

김동연 경기도지사는 "이번 주 갑작스러운 기온 하강에 따라 피해가 우려된다. 노약자 등 취약계층에 대한 안전은 물론 농축업 등 산업분야까지 세심하게 살펴 피해를
----------------------------------------------------------------------------------------------------
Document 3:

이천, 안성, 화성, 광주 등 경기남부지역을 비롯한 21개 지역에는 한파주의보를 발표됐다.
----------------------------------------------------------------------------------------------------
Document 4:

기상청은 이날 오후 9시를 기해 경기도 전역에 한파특보를 발효할 예정이다. 한파주의보가 내려졌던 연천·포천·가평·파주 등 4개 시·군은 한파경보로 변경되고,
----------------------------------------------------------------------------------------------------
Document 5:

한파 피해 최소화를 위해 ▲노인, 노숙인 등 취약계층 안전확인 강화 ▲지역자율방재단, 이·통장 등 협업을 통한 한파쉼터 운영상황 등 점검 ▲농작물 냉해 및 가축
----------------------------------------------------------------------------------------------------
Document 6:

등 점검 ▲농작물 냉해 및 가축 동사 방지 등

In [25]:
!pip install langchain_classic

## Cohere Rerank model - Reranker

#### ContextualCompressionRetriever 활용하여 Reranker 추가하기

In [27]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-multilingual-v3.0", top_n = 5) # rerank-english-v3.0
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "한파에 대비해야할 것에는 어떤게 있나요?"
)
pretty_print_docs(compressed_docs)

Document 1:

등 점검 ▲농작물 냉해 및 가축 동사 방지 등 사전 대비 ▲야외활동 자제, 부모님께 안부전화 하기 등 한파 행동요령과 안전수칙 적극 홍보를 시군에 요청했다.
----------------------------------------------------------------------------------------------------
Document 2:

아울러 수도계량기·노출 수도관·보일러 등의 보온 상태를 점검해 동파에 대비하고, 온실과 축사에 난방장치를 가동해 농작물과 가축의 동사를 방지해야 한다.
----------------------------------------------------------------------------------------------------
Document 3:

도는 재난안전대책본부 1단계를 가동해 복지·상하수 분야 등 6개 반 13개 부서 13명이 분야별 대응하며 한파에 대비할 방침이다.
----------------------------------------------------------------------------------------------------
Document 4:

한파 피해 최소화를 위해 ▲노인, 노숙인 등 취약계층 안전확인 강화 ▲지역자율방재단, 이·통장 등 협업을 통한 한파쉼터 운영상황 등 점검 ▲농작물 냉해 및 가축
----------------------------------------------------------------------------------------------------
Document 5:

[수원=뉴시스] 이병희 기자 = 경기도가 도내 31개 시군 전역에 예보된 한파특보에 따라 8일 오후 1시부터 재난안전대책본부 1단계를 가동, 선제 대응에


## RAG

[Langchain Docs : create_stuff_documents_chain](https://python.langchain.com/api_reference/langchain/chains/langchain.chains.combine_documents.stuff.create_stuff_documents_chain.html)

[Langchain Docs : create_retrieval_chain](https://python.langchain.com/api_reference/langchain/chains/langchain.chains.retrieval.create_retrieval_chain.html)

In [31]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_cohere import ChatCohere

llm = ChatCohere()

system_prompt = (
    "Use the given context to answer the question in Korean. "
    "If you don't know the answer, say you don't know. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(compression_retriever, combine_docs_chain)

In [32]:
query = "한파에 대비해야할 것에는 어떤게 있나요?"
response = chain.invoke({"input": query})

In [33]:
response['input']

'한파에 대비해야할 것에는 어떤게 있나요?'

In [34]:
response['context']

[Document(metadata={'source': 'sample.txt', 'relevance_score': 0.96295285}, page_content='등 점검 ▲농작물 냉해 및 가축 동사 방지 등 사전 대비 ▲야외활동 자제, 부모님께 안부전화 하기 등 한파 행동요령과 안전수칙 적극 홍보를 시군에 요청했다.'),
 Document(metadata={'source': 'sample.txt', 'relevance_score': 0.9568766}, page_content='아울러 수도계량기·노출 수도관·보일러 등의 보온 상태를 점검해 동파에 대비하고, 온실과 축사에 난방장치를 가동해 농작물과 가축의 동사를 방지해야 한다.'),
 Document(metadata={'source': 'sample.txt', 'relevance_score': 0.6509112}, page_content='도는 재난안전대책본부 1단계를 가동해 복지·상하수 분야 등 6개 반 13개 부서 13명이 분야별 대응하며 한파에 대비할 방침이다.'),
 Document(metadata={'source': 'sample.txt', 'relevance_score': 0.484906}, page_content='한파 피해 최소화를 위해 ▲노인, 노숙인 등 취약계층 안전확인 강화 ▲지역자율방재단, 이·통장 등 협업을 통한 한파쉼터 운영상황 등 점검 ▲농작물 냉해 및 가축'),
 Document(metadata={'source': 'sample.txt', 'relevance_score': 0.09186979}, page_content='[수원=뉴시스] 이병희 기자 = 경기도가 도내 31개 시군 전역에 예보된 한파특보에 따라 8일 오후 1시부터 재난안전대책본부 1단계를 가동, 선제 대응에')]

In [35]:
print(response['answer'])

한파에 대비해야 할 주요 사항은 다음과 같습니다:

1. **농작물 및 가축 보호**:  
   - 농작물 냉해 방지 및 가축 동사 방지를 위해 온실과 축사에 난방장치를 가동해야 합니다.  
   - 사전 대비 조치를 철저히 해야 합니다.

2. **수도 시설 동파 방지**:  
   - 수도계량기, 노출 수도관, 보일러 등의 보온 상태를 점검하여 동파를 예방해야 합니다.

3. **야외활동 자제 및 안전수칙 준수**:  
   - 야외활동을 자제하고, 한파 행동요령과 안전수칙을 적극 홍보하며 실천해야 합니다.  
   - 부모님이나 주변 분들에게 안부 전화를 드리는 것도 중요합니다.

4. **취약계층 안전 확인**:  
   - 노인, 노숙인 등 취약계층의 안전을 확인하고, 한파쉼터 운영 상황을 점검해야 합니다.  
   - 지역자율방재단, 이·통장 등과의 협업을 통해 대응 체계를 강화합니다.

5. **재난안전대책본부 가동**:  
   - 재난안전대책본부를 가동하여 복지, 상하수 분야 등 분야별 대응 체계를 구축하고 한파에 대비합니다.

이러한 조치들을 통해 한파 피해를 최소화할 수 있습니다.
